In [1]:
from collections import Counter, defaultdict, namedtuple
from copy import deepcopy
import functools
import inspect
import json
import os
from pathlib import Path
import pickle
from pprint import pp, pprint, pformat
import re
import sys
import time
from typing import Dict, List

from jsonschema import validate
import numpy as np
import pandas as pd
import plotly.express as px
import xmltodict

from colorutils import Color

from dotenv import load_dotenv
from jinja2 import Environment, FileSystemLoader, Template
import textwrap
from tqdm.auto import tqdm
# from tqdm import tqdm

import openai
from openai import OpenAI

from aic_nlp_utils.json import read_jsonl, read_json, write_json, write_jsonl, process_to_jsonl
from aic_nlp_utils.pycfg import parse_pycfg_args, read_pycfg
%load_ext autoreload
%autoreload 2

from prompt_opt.optimizers.predict_evaluate import get_candidate_score, rank_candidates
from prompt_opt.utils import *

sys.path.append("/home/drchajan/devel/python/FC/automated-fact-checking")

os.environ['VLLM_WORKER_MULTIPROC_METHOD']='spawn'
load_dotenv()

True

In [15]:
client = OpenAI(base_url="http://g04:8333/v1")
model_name = client.models.list().data[0].id
model_name

'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'

In [15]:
import logging
logging.basicConfig(level=logging.DEBUG)

In [16]:
import requests
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

tool_functions = {"get_weather": get_weather}

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for provided coordinates in celsius.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

messages = [{"role": "user", "content": "What's the weather like in Paris today?"}]

response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

tool_call = response.choices[0].message.tool_calls[0]
print(f"Function called: {tool_call.function.name}")
print(f"Arguments: {tool_call.function.arguments}")
print(f"Calls: {response.choices[0].message.tool_calls[0]}")
tool_res = get_weather(**json.loads(tool_call.function.arguments))
print(f"Result: {tool_res}")


IndexError: list index out of range

In [10]:
messages.append(response.choices[0].message)  # append model's function call message
messages.append({                               # append result message
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(tool_res)
})

response2 = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
)

In [14]:
print(response2.choices[0].message.content)

The current temperature in Paris is 10.2 degrees Celsius.


In [12]:
response = client.chat.completions.create(
    # model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    model=model_name,
    messages=[{"role": "user", "content": 'Define Churchill as JSON. Use format: {"name": ..., "surname": ..., "birth_date": ...}'}],
    # messages=[{"role": "user", "content": meta[0]["messages"][0]["content"]}],
    # max_tokens=500,
    temperature= 0.6,
    frequency_penalty= 0.05
)


# Print the generated text
print(response.choices[0].message.content)
# print(response.choices[0].message.reasoning_content)

Here's the definition of Winston Churchill in JSON format:

```json
{
  "name": "Winston",
  "surname": "Churchill",
  "birth_date": "November 30, 1874",
  "death_date": "January 24, 1965",
  "nationality": "British",
  " occupation": "Politician, Writer, Painter",
  "notable_for": "Leading Britain to victory in World War II, Prime Minister of the United Kingdom"
}
```

Note: The "death_date" has been included as it is also an important piece of information about Winston Churchill.
